<a href="https://colab.research.google.com/github/pikey-msc/AprendizMaquina/blob/main/2025-2/Ensambles/Stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🧠 Entrenamiento del Stacking

El **stacking** (apilamiento) es un método de ensamble basado en un enfoque de **aprendizaje supervisado en dos niveles**. No se basa en un único algoritmo, sino en una **arquitectura de entrenamiento secuencial en capas**, en donde el modelo de nivel superior (**meta-modelo**) aprende a combinar las salidas de varios modelos base (**modelos de primer nivel**).

### 🔁 Procedimiento paso a paso

1. **Entrenamiento de modelos base (`nivel-0`)**:

   Se definen $L$ modelos base: $f_1, f_2, \dots, f_L$.

   Para evitar overfitting, se usa validación cruzada (por ejemplo, $K$-fold) para generar **predicciones out-of-fold**. Esto quiere decir que:
   
   - A cada modelo $f_\ell$ se le entrena con $K-1$ *folds*, y se predice sobre el *fold* restante.
   - Este procedimiento se repite para cada fold.
   - Así se obtiene un vector de predicciones para cada modelo, sin usar los mismos datos con los que se entrenó.

2. **Construcción del conjunto de entrenamiento para el meta-modelo (`nivel-1`)**:

   Se construye una nueva matriz de características $Z$ de tamaño $n \times L$, donde:

   $$
   Z_{i,\ell} = f_\ell^{(-k)}(x_i)
   $$

   - $f_\ell^{(-k)}$ indica que el modelo fue entrenado sin el fold que contiene $x_i$.
   - $x_i$ es la $i$-ésima observación.
   - Esta matriz $Z$ se usa como entrada para el meta-modelo.

3. **Entrenamiento del meta-modelo (`nivel-1`)**:

   Se entrena un nuevo modelo $g$ (por ejemplo, regresión logística, árbol, etc.) sobre el conjunto $\{Z, y\}$:

   $$
   \hat{y}_i = g(Z_{i,1}, Z_{i,2}, \dots, Z_{i,L})
   $$

   Este modelo aprende **cómo combinar las predicciones de los modelos base** para obtener la mejor predicción posible.

4. **Predicción sobre nuevos datos**:

   - Los nuevos datos $\mathbf{x}^\ast$ son pasados por cada modelo base: $f_1(\mathbf{x}^\ast), f_2(\mathbf{x}^\ast), \dots, f_L(\mathbf{x}^\ast)$.
   - Luego, estas predicciones se utilizan como entrada del meta-modelo: $g(f_1(\mathbf{x}^\ast), \dots, f_L(\mathbf{x}^\ast))$.

### 📌 Observaciones

- El modelo meta **se entrena sobre predicciones, no sobre los datos originales**.
- Los modelos base se deben volver a entrenar con todo el conjunto de entrenamiento al final para usarlos en predicción.
- El stacking no asume independencia entre predictores como Bagging o Boosting, sino que se enfoca en **aprovechar la diversidad** entre modelos base.

### 📈 Función objetivo

El meta-modelo resuelve un problema de aprendizaje supervisado tradicional:

$$
g = \arg\min_{\tilde{g}} \sum_{i=1}^n \mathcal{L}(y_i, \tilde{g}(Z_i))
$$

Donde $\mathcal{L}$ es una función de pérdida apropiada al problema (por ejemplo, error cuadrático medio para regresión, log-loss o entropía cruzada para clasificación binaria).

---

               

## 🔧 ¿Qué funciones $g$ se pueden utilizar como meta-modelo en Stacking?

La función $g$ en el modelo de stacking representa al **meta-modelo** que combina las predicciones de los modelos base. Su elección debe ser coherente con el tipo de variable de salida $y$ (clase o continua).

---

### 📊 Para Clasificación (salida discreta)

- **Regresión logística** (`LogisticRegression`):
  - Muy común como modelo $g$ por su simplicidad e interpretabilidad.
  - Se recomienda si los modelos base producen probabilidades.

- **Árbol de decisión / Random Forest** (`DecisionTreeClassifier`, `RandomForestClassifier`):
  - Capturan no linealidades en las predicciones de los modelos base.
  - Útiles si hay interacción compleja entre predictores.

- **K-Nearest Neighbors (KNN)** (`KNeighborsClassifier`):
  - Considera similitudes entre predicciones de los modelos base.
  - Más costoso computacionalmente.

- **Support Vector Machine (SVM)** (`SVC` con `probability=True`):
  - Ideal cuando las predicciones de los modelos base no son linealmente separables.
  - Puede ser sensible al escalado.

- **Redes Neuronales** (`MLPClassifier`):
  - Permiten aprender combinaciones altamente no lineales.
  - Requieren más datos y regularización.

---

### 📈 Para Regresión (salida continua)

- **Regresión lineal** (`LinearRegression`, `Ridge`, `Lasso`):
  - Simples y rápidas, útiles como baseline.
  - `Ridge` y `Lasso` ayudan a evitar sobreajuste si hay muchas predicciones base.

- **Árboles de regresión / Gradient Boosting** (`DecisionTreeRegressor`, `GradientBoostingRegressor`):
  - Capturan relaciones complejas entre predicciones.
  - Eficientes si las salidas base tienen interacciones.

- **KNN para regresión** (`KNeighborsRegressor`):
  - Útil si los modelos base producen resultados similares para casos cercanos.

- **SVM para regresión** (`SVR`):
  - Requiere tuning cuidadoso, pero útil para relaciones no lineales.

- **Redes neuronales (perceptrón multicapa)** (`MLPRegressor`):
  - Potente para detectar patrones complejos entre predicciones de modelos base.

---

### ✅ Buenas prácticas al elegir $g$

- **Empieza simple**: usar `LogisticRegression` o `LinearRegression` como baseline.
- **Evita overfitting**: si usas un meta-modelo muy flexible (e.g. MLP), asegúrate de tener suficientes datos y usar validación cruzada.
- **Haz tuning por separado**: el meta-modelo también puede tener hiperparámetros que afectan el desempeño final del stacking.
- **Escalado**: algunos modelos como `SVM` o `MLP` requieren que las entradas (las predicciones de los modelos base) estén normalizadas.

---

### 📌 Observación final

En stacking, el meta-modelo **no tiene acceso a las características originales** ($X$), sólo a las **predicciones de los modelos base**. Sin embargo, en algunas variantes extendidas (e.g. *stacking generalizado*), sí se permite incluir tanto $X$ como las salidas base en $g$.



### Ejemplo stacking manual

In [12]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# 1. Crear datos sintéticos
X, y = make_classification(n_samples=1000, n_features=10)

# 2. Definir modelos base
base_models = [DecisionTreeClassifier(), GaussianNB()]
meta_model = LogisticRegression()

# 3. Crear validación cruzada para out-of-fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
meta_features = np.zeros((X.shape[0], len(base_models)))

for i, model in enumerate(base_models):
    for train_idx, val_idx in kf.split(X):
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        meta_features[val_idx, i] = preds

# 4. Entrenar meta-modelo con predicciones de los modelos base
meta_model.fit(meta_features, y)

# 5. Entrenar modelos base con todos los datos
for model in base_models:
    model.fit(X, y)

# 6. Predicción con stacking
def predict_stacking(X_new):
    base_preds = np.column_stack([model.predict(X_new) for model in base_models])
    return meta_model.predict(base_preds)

# 7. Evaluar
y_pred = predict_stacking(X)
acc = accuracy_score(y, y_pred)
f1 = f1_score(y, y_pred)
cm = confusion_matrix(y, y_pred)

print("Accuracy del modelo stacking:", acc)
print("F1-Score del modelo stacking:", f1)
print("Matriz de confusión:")
print(cm)


Accuracy del modelo stacking: 0.941
F1-Score del modelo stacking: 0.941871921182266
Matriz de confusión:
[[463  39]
 [ 20 478]]


### Ejemplo de stacking con scikit-learn

In [10]:
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# 1. Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Definir modelos
estimators = [
    ('dt', DecisionTreeClassifier()),
    ('nb', GaussianNB())
]
final_estimator = LogisticRegression()

# 3. Definir stacking classifier
stack = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5
)

# 4. Entrenar
stack.fit(X_train, y_train)

# 5. Predicción
y_pred_stack = stack.predict(X_test)

# 6. Evaluación
acc = accuracy_score(y_test, y_pred_stack)
f1 = f1_score(y_test, y_pred_stack)
cm = confusion_matrix(y_test, y_pred_stack)

print("Accuracy con sklearn StackingClassifier:", acc)
print("F1-Score:", f1)
print("Matriz de confusión:")
print(cm)


Accuracy con sklearn StackingClassifier: 0.85
F1-Score: 0.8623853211009175
Matriz de confusión:
[[76 13]
 [17 94]]


### Ejemplo comparando distintos modelos meta

In [7]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix


# 1. Crear dataset sintético
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2,
                           n_informative=10, random_state=42)

# 2. Separar en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Modelos base comunes
base_estimators = [
    ('tree', DecisionTreeClassifier(random_state=42)),
    ('nb', GaussianNB())
]

# 4. Diferentes meta-modelos para comparar
meta_models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'MLPClassifier': MLPClassifier(hidden_layer_sizes=(20,), max_iter=1000, random_state=42)
}

# 5. Evaluación de cada modelo de stacking
results_with_cm = []
for name, meta in meta_models.items():
    stack_model = StackingClassifier(
        estimators=base_estimators,
        final_estimator=meta,
        cv=5
    )
    stack_model.fit(X_train, y_train)
    y_pred = stack_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)

    results_with_cm.append({
        'Meta-Modelo': name,
        'Accuracy': acc,
        'F1-Score': f1,
        'Matriz Confusión': cm
    })

# Convertir a DataFrame con matriz de confusión como string para visualización
results_df_cm = pd.DataFrame(results_with_cm)
results_df_cm['Matriz Confusión'] = results_df_cm['Matriz Confusión'].apply(lambda x: f"{x.tolist()}")

from IPython.display import display
display(results_df_cm)



,Meta-Modelo,Accuracy,F1-Score,Matriz Confusión
0,LogisticRegression,0.873333,0.872483,"[[132, 21], [17, 130]]"
1,RandomForest,0.823333,0.823920,"[[123, 30], [23, 124]]"
2,MLPClassifier,0.873333,0.872483,"[[132, 21], [17, 130]]"


# 🧠 Stacking Generalizado (Generalized Stacking)

El **Stacking Generalizado** es una extensión del método clásico de *stacking* donde el modelo meta (**meta-modelo**) no solo recibe las predicciones de los modelos base, sino que también puede tener acceso a las **características originales** del conjunto de datos ($\mathbf{X}$).

---

## 🔁 Comparación con Stacking clásico

En el stacking tradicional:

- Se entrena un conjunto de modelos base $f_1, f_2, \dots, f_L$ sobre el conjunto original $(\mathbf{X}, \mathbf{y})$.
- Las predicciones $\hat{y}_i^{(f_\ell)}$ de los modelos base se apilan para formar una nueva matriz $\mathbf{Z}$.
- Un meta-modelo $g$ se entrena sobre $(\mathbf{Z}, \mathbf{y})$, es decir, **sin ver $\mathbf{X}$**.

En el stacking generalizado, el meta-modelo $g$ se entrena sobre una matriz extendida:

$$
\mathbf{Z}^{\text{gen}} = [\mathbf{X} \mid f_1(\mathbf{X}) \mid f_2(\mathbf{X}) \mid \cdots \mid f_L(\mathbf{X})]
$$

Esto quiere decir que el meta-modelo recibe tanto:

- Las **características originales** $\mathbf{X}$,
- Como las **predicciones de los modelos base** $f_1(\mathbf{X}), \dots, f_L(\mathbf{X})$.

---

## 📐 Representación formal

Dado:

- $\mathbf{X} \in \mathbb{R}^{n \times m}$: conjunto de características originales,
- $f_\ell(\mathbf{X})$: predicción del modelo base $\ell$ (puede ser clase, probabilidad, o valor continuo),

Entonces la entrada al meta-modelo $g$ es:

$$
g\left( \mathbf{x}_i, f_1(\mathbf{x}_i), f_2(\mathbf{x}_i), \dots, f_L(\mathbf{x}_i) \right)
$$

Este modelo busca resolver:

$$
\hat{y}_i = g([\mathbf{x}_i, f_1(\mathbf{x}_i), \dots, f_L(\mathbf{x}_i)])
$$

---

## 🧪 ¿Por qué usar Stacking Generalizado?

✅ **Ventajas:**

- El meta-modelo puede capturar **interacciones** entre las variables originales y las predicciones de los modelos base.
- Puede corregir errores sistemáticos de los modelos base usando las variables originales.

⚠️ **Desventajas:**

- Mayor riesgo de **sobreajuste**, especialmente si el meta-modelo es muy complejo.
- Mayor costo computacional y complejidad en la arquitectura del modelo.

---

## 🧩 Implementación práctica

En `sklearn`, no existe directamente un `StackingClassifier` que soporte `X + predicciones base` como entrada para el meta-modelo, pero se puede construir manualmente:

1. Entrenar los modelos base.
2. Generar las predicciones sobre un conjunto de validación.
3. Concatenar esas predicciones con las variables originales.
4. Entrenar el meta-modelo sobre esa matriz extendida.

---

## 🔎 Consideraciones

- Es útil cuando se sospecha que las predicciones base **no capturan toda la información útil de las variables originales**.
- Si se usa regresión logística como meta-modelo, incluir $\mathbf{X}$ permite modelar directamente relaciones lineales que los modelos base omitieron.
- Requiere **cuidadosa validación cruzada**, para evitar "contaminación de datos" entre las predicciones base y las verdaderas.

---

## 📚 Referencias

- Wolpert, D. H. (1992). *Stacked Generalization*. Neural Networks.
- Breiman, L. (1996). *Stacked regressions*. Machine Learning.



### Ejemplo de Stacking Generalizado manual y con librerías - Clasificación

In [14]:


from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.ensemble import StackingClassifier
import numpy as np
import pandas as pd

# 1. Crear dataset sintético
X, y = make_classification(n_samples=1000, n_features=10, n_informative=6, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

### --- Stacking Generalizado Manual ---
kf = KFold(n_splits=5, shuffle=True, random_state=42)
base_models = [DecisionTreeClassifier(), GaussianNB()]
meta_X_train = np.zeros((X_train.shape[0], len(base_models)))
meta_X_test = np.zeros((X_test.shape[0], len(base_models)))

# Out-of-fold para entrenamiento
for i, model in enumerate(base_models):
    meta_column = np.zeros(X_train.shape[0])
    for train_idx, val_idx in kf.split(X_train):
        model.fit(X_train[train_idx], y_train[train_idx])
        meta_column[val_idx] = model.predict(X_train[val_idx])
    meta_X_train[:, i] = meta_column
    model.fit(X_train, y_train)  # Reentrenar en todo el entrenamiento
    meta_X_test[:, i] = model.predict(X_test)

# Concatenar X + predicciones base (stacking generalizado)
extended_X_train = np.concatenate([X_train, meta_X_train], axis=1)
extended_X_test = np.concatenate([X_test, meta_X_test], axis=1)

# Meta-modelo sobre stacking generalizado
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(extended_X_train, y_train)
y_pred_manual = meta_model.predict(extended_X_test)

# Métricas
acc_manual = accuracy_score(y_test, y_pred_manual)
f1_manual = f1_score(y_test, y_pred_manual)
cm_manual = confusion_matrix(y_test, y_pred_manual)

### --- Stacking usando sklearn ---
stacking = StackingClassifier(
    estimators=[('dt', DecisionTreeClassifier()), ('nb', GaussianNB())],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)
stacking.fit(X_train, y_train)
y_pred_sklearn = stacking.predict(X_test)

# Métricas sklearn stacking
acc_sk = accuracy_score(y_test, y_pred_sklearn)
f1_sk = f1_score(y_test, y_pred_sklearn)
cm_sk = confusion_matrix(y_test, y_pred_sklearn)

# Resultados
results = {
    'Método': ['Stacking Generalizado Manual', 'Stacking sklearn'],
    'Accuracy': [acc_manual, acc_sk],
    'F1-Score': [f1_manual, f1_sk],
    'Matriz Confusión': [cm_manual.tolist(), cm_sk.tolist()]
}

results_df = pd.DataFrame(results)
from IPython.display import display
display(results_df)



,Método,Accuracy,F1-Score,Matriz Confusión
0,Stacking Generalizado Manual,0.846667,0.843537,"[[130, 32], [14, 124]]"
1,Stacking sklearn,0.840000,0.838926,"[[127, 35], [13, 125]]"
